In [1]:
import os 

path= "/Users/jpardo/Desktop/Proyectos/AI4LABOUR/ai4labour_preprocess/Mejoras/results"


#archivos_csv = [i for i in os.listdir(path) if i.endswith('.csv')]

In [2]:
import os 
os.chdir(path)

# Ranking, sweep, best model

## 1. Métricas

### **Precisión @k (P\@k)**

* Intuición: de los *k* primeros resultados que devuelve el modelo, ¿cuántos son relevantes?
* Fórmula:

$$
P@k = \frac{\# \text{relevantes en top-k}}{k}
$$



### **Recall @k (R\@k)**

* Intuición: de todos los relevantes existentes, ¿cuántos recuperé en los *k* primeros?
* Fórmula:

$$
R@k = \frac{\# \text{relevantes en top-k}}{\# \text{total de relevantes}}
$$



### **nDCG @k (Normalized Discounted Cumulative Gain)**

* Intuición: no sólo importa si recupero relevantes, también importa **la posición** en que los coloco (los primeros valen más).
* Se calcula en tres pasos:

  1. **DCG\@k** (Discounted Cumulative Gain):

  $$
  DCG@k = \sum_{i=1}^k \frac{rel_i}{\log_2(i+1)}
  $$

  donde $rel_i = 1$ si el resultado en posición $i$ es relevante, 0 si no.
  2\. **IDCG\@k** = DCG ideal (orden perfecto).
  3\. **nDCG\@k** = normalización:

  $$
  nDCG@k = \frac{DCG@k}{IDCG@k}
  $$



### **MRR (Mean Reciprocal Rank)**

* Intuición: mide en qué posición aparece el **primer relevante**. Cuanto más arriba, mejor.
* Fórmula:

$$
MRR = \frac{1}{|Q|} \sum_{q=1}^{|Q|} \frac{1}{rank_q}
$$

donde $rank_q$ es la posición del primer relevante para la query $q$.



### **Recall\@1** (columna aparte)

* Es una forma concreta de **R\@1**: ¿qué porcentaje de queries recuperan al menos un relevante en la primera posición?

$$
Recall@1 = \frac{\# \text{queries cuyo top-1 es relevante}}{|Q|}
$$



### **coverage\_queries**

* Intuición: de todas las queries evaluadas, ¿para cuántas el modelo devolvió al menos un candidato?
* Fórmula:

$$
coverage = \frac{\# \text{queries con ≥1 candidato}}{\# \text{queries totales}}
$$

En tu caso vale 1.0 → siempre hay salida para cada query.



### **F1**

* Media armónica de precisión y recall:

$$
F1 = 2 \cdot \frac{precision \cdot recall}{precision + recall}
$$

## 2. ¿Qué significa el símbolo “@k”?

El **@k** indica el **corte en el ranking**.
Ejemplo: P\@1 = precisión considerando **sólo el primer resultado**; P\@5 = precisión considerando los 5 primeros.

En tu tabla tienes P\@1, P\@3, P\@5, P\@10, y lo mismo para Recall y nDCG.
Esto permite ver si el modelo es bueno en los **primeros puestos (alta prioridad)** o si mejora cuando se considera un top más amplio.



In [3]:
import pandas as pd

df_rank = pd.read_csv("ranking_summary.csv")
df_sweep = pd.read_csv("threshold_sweep.csv")
df_best = pd.read_csv("best_thresholds.csv")

display(df_rank.sort_values("nDCG@10", ascending=False))
display(df_best)


,P@1,P@3,P@5,P@10,R@1,R@3,R@5,R@10,nDCG@1,nDCG@3,nDCG@5,nDCG@10,MRR,Recall@1,coverage_queries,model
0,0.6,0.366667,0.28,0.16,0.30,0.55,0.70,0.80,0.6,0.672629,0.746476,0.785372,0.741667,0.30,1.0,sentence-transformers/sentence-t5-base
1,0.6,0.333333,0.30,0.15,0.30,0.50,0.75,0.75,0.6,0.597037,0.755323,0.755323,0.728333,0.30,1.0,sentence-transformers/all-mpnet-base-v2
2,0.5,0.366667,0.26,0.16,0.25,0.55,0.65,0.80,0.5,0.624408,0.693882,0.753885,0.689286,0.25,1.0,sentence-transformers/all-distilroberta-v1
3,0.5,0.266667,0.20,0.13,0.25,0.40,0.50,0.65,0.5,0.601778,0.668566,0.747868,0.691667,0.25,1.0,sentence-transformers/distiluse-base-multiling...
4,0.4,0.333333,0.24,0.14,0.20,0.50,0.60,0.70,0.4,0.501778,0.554592,0.607937,0.566667,0.20,1.0,sentence-transformers/all-MiniLM-L6-v2
5,0.4,0.266667,0.24,0.15,0.20,0.40,0.60,0.75,0.4,0.383944,0.503545,0.585796,0.518333,0.20,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
6,0.5,0.233333,0.16,0.10,0.25,0.35,0.40,0.50,0.5,0.461315,0.504382,0.553727,0.535000,0.25,1.0,sentence-transformers/paraphrase-albert-small-v2
7,0.3,0.233333,0.18,0.10,0.15,0.35,0.45,0.50,0.3,0.426186,0.507939,0.543560,0.461667,0.15,1.0,sentence-transformers/paraphrase-MiniLM-L3-v2
8,0.2,0.133333,0.12,0.10,0.10,0.20,0.30,0.50,0.2,0.255065,0.317470,0.431620,0.329286,0.10,1.0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...
9,0.1,0.066667,0.04,0.03,0.05,0.10,0.10,0.15,0.1,0.163093,0.163093,0.198714,0.166667,0.05,1.0,embedding-data/deberta-sentence-transformer


,model,best_thr,f1,precision,recall,coverage
0,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...,-0.5,0.266667,0.400000,0.20,1.0
1,/home/jovyan/Joel_Pardo/AI4LABOUR/Mejoras/resu...,-0.5,0.133333,0.200000,0.10,1.0
2,embedding-data/deberta-sentence-transformer,0.9,0.074074,0.142857,0.05,0.7
3,sentence-transformers/all-MiniLM-L6-v2,-0.5,0.266667,0.400000,0.20,1.0
4,sentence-transformers/all-distilroberta-v1,-0.5,0.333333,0.500000,0.25,1.0
5,sentence-transformers/all-mpnet-base-v2,-0.5,0.400000,0.600000,0.30,1.0
6,sentence-transformers/distiluse-base-multiling...,-0.5,0.333333,0.500000,0.25,1.0
7,sentence-transformers/paraphrase-MiniLM-L3-v2,-0.5,0.200000,0.300000,0.15,1.0
8,sentence-transformers/paraphrase-albert-small-v2,0.6,0.357143,0.625000,0.25,0.8
9,sentence-transformers/sentence-t5-base,-0.5,0.400000,0.600000,0.30,1.0






## 3. Cómo se ha llegado hasta aquí en tu pipeline

Basándonos en los bloques A–H que revisamos juntos:

1. **Carga datos (bloque A)**: se cargaron las DWAs (detailed work activities) desde `dwas.csv`.

2. **Embeddings (bloque C)**: para cada modelo en `CANDIDATE_MODELS` (ej. MiniLM, MPNet, DistilRoberta, T5-base, DeBERTa, etc.), se generaron embeddings normalizados.

3. **Vecinos (bloque D+E)**: con cada embedding, se calculó la similitud coseno y se construyeron rankings de “vecinos” (DWAs más cercanas). Esos rankings se guardaron en CSV (`neighbors_*`).

4. **Evaluación (bloque F)**:

   * Se usó el **GOLD** (un conjunto de queries con positivos y negativos definidos manualmente).
   * Para cada query del GOLD, se compararon los rankings generados por cada modelo contra la lista de “positivos” esperados.
   * Se calcularon las métricas de ranking (P\@k, R\@k, nDCG\@k, MRR).
   * Además, se exploraron diferentes **thresholds** de score → por cada umbral, se calculó precisión, recall, F1, y coverage. Eso alimenta `threshold_sweep.csv` y `best_thresholds.csv`.

5. **Reporte (bloque G)**: se seleccionó un modelo ganador según las métricas (por defecto nDCG\@10 y desempate por Recall\@1). También se generaron ficheros de errores (TP, FP, FN).

6. **Visualización (bloque H)**: se graficaron curvas y distribuciones de scores.



##  4. Resultados

* Modelos como **MPNet-base** y **T5-base** destacan en nDCG\@10 (\~0.75–0.78) y MRR (\~0.74).
* El **Recall\@1** máximo es 0.30 → sólo un 30% de queries tienen un relevante en la primera posición → margen de mejora importante.
* En thresholds, se ve que modelos como **T5-base** y **MPNet** logran el mejor F1 (0.40).
* **DeBERTa** tiene buen coverage pero F1 muy bajo (muchos FP → baja precisión).

# Fasos positivos/ negativos


##  1. Estructura de la tabla superior (falsos positivos / cerca del umbral)

Columnas clave:

* **q\_label** → la query del GOLD (ej. `"Evaluate quality of materials or products."`).
* **cand\_label** → el candidato recuperado por el modelo como vecino más cercano.
* **score** → similitud coseno (o score compuesto) entre query y candidato.
* **delta\_top1\_top2** → diferencia de score entre el **top-1** y el **top-2**.

  * Si es muy pequeño, el modelo estaba indeciso → alta ambigüedad.
* **sim** → valor puro de similitud coseno (antes de penalizaciones).
* **z** → z-score de la similitud respecto a la distribución de scores de esa query.

  * Indica cuán “anómalo” o fuerte es el match.
* **lenpen** (length penalty) → penalización por diferencias de longitud entre query y candidato.

  * En tu caso 0 o 0.2.
* **stopjac** → Jaccard sobre stopwords. Si es alto, probablemente hay emparejamiento artificial por palabras vacías comunes.
* **mnn** → indica si hay **Mutual Nearest Neighbor** (el emparejamiento es recíproco en ambos sentidos). True → mayor robustez.

Interpretación:
Son **casos donde el modelo dio un score alto**, pero manualmente no estaban en el set de “positivos” (probables **falsos positivos** o “near-threshold”).
Ejemplo: `"Evaluate quality of materials or products."` ↔ `"Assess product or process usefulness."` (score=0.91). Suena razonable, pero si no estaba en el GOLD → se marca como error.



## 2. Estructura de la tabla inferior (falsos negativos)

Columnas clave:

* **q\_label** → query evaluada.
* **missed\_positive** → candidato que estaba marcado como positivo en el GOLD pero no fue recuperado correctamente por el modelo. (**Falso negativo**).
* **top1\_cand** → el candidato que el modelo sí puso como top-1.
* **top1\_score** → score de ese top-1.
* **delta\_top1\_top2** → diferencia entre el top-1 y el top-2 del modelo.

Interpretación:

* Ejemplo: `"Design medical devices or appliances."` tenía como positivo `"Design electromechanical equipment or systems."`, pero el modelo se fue con `"Design electronic or computer equipment or instruments."`.

  * Ambos están cerca semánticamente, pero no coincide exactamente con el GOLD → **FN**.
* `"Estimate operational costs."` esperaba `"Estimate labor requirements."`, pero el modelo se quedó con `"Estimate cost or material requirements."`.

In [4]:
df_fp = pd.read_csv("errors_fp.csv")
df_fn = pd.read_csv("errors_fn.csv")
display(df_fp.sample(5, replace=True))
display(df_fn.head())


,q_label,cand_label,score,delta_top1_top2,sim,z,lenpen,stopjac,mnn
3,Prepare procedural documents.,Prepare proposal documents.,0.873937,0.007394,0.923937,3.424706,0.00,0.500000,True
3,Prepare procedural documents.,Prepare proposal documents.,0.873937,0.007394,0.923937,3.424706,0.00,0.500000,True
1,Estimate operational costs.,Estimate cost or material requirements.,0.882767,0.007005,0.911933,2.786119,0.25,0.166667,True
1,Estimate operational costs.,Estimate cost or material requirements.,0.882767,0.007005,0.911933,2.786119,0.25,0.166667,True
1,Estimate operational costs.,Estimate cost or material requirements.,0.882767,0.007005,0.911933,2.786119,0.25,0.166667,True


,q_label,missed_positive,top1_cand,top1_score,delta_top1_top2
0,Design medical devices or appliances.,Design electromechanical equipment or systems.,Design electronic or computer equipment or ins...,0.919826,0.001440
1,Estimate operational costs.,Estimate labor requirements.,Estimate cost or material requirements.,0.882767,0.007005
2,Evaluate quality of materials or products.,Inspect finished products to locate flaws.,Assess product or process usefulness.,0.913625,0.035291
3,Prepare procedural documents.,Document organizational or operational procedu...,Prepare proposal documents.,0.873937,0.007394


## 3. Métricas auxiliares

* **score** / **sim** → indican la “confianza” del modelo en el match.
* **delta\_top1\_top2** → útil como señal: si es pequeño, significa que había empate → el error puede no ser grave (modelo confundido entre dos candidatos muy similares).
* **z** → normaliza por query; si es alto, el match es fuerte comparado con el resto.
* **lenpen** y **stopjac** → heurísticas para reducir sesgos (no aplican siempre, pero ayudan a explicar errores).
* **mnn=True** → significa que la similitud es recíproca, lo cual suele ser un buen indicador de match real.



## 4. Cómo se llegó a estas tablas

1. Se tomó el **modelo ganador** del bloque F (según nDCG y Recall\@1).
2. Se aplicó el **umbral óptimo** (`best_thr`) para decidir qué matches aceptar o rechazar.
3. Se comparó cada predicción contra el GOLD.

   * Si el candidato estaba en positivos → TP.
   * Si no estaba en positivos pero pasó el umbral → FP.
   * Si un positivo nunca apareció en los recuperados → FN.
4. Se extrajeron subconjuntos:

   * **errors\_fp.csv** → falsos positivos (lo que ves arriba, primera tabla).
   * **errors\_fn.csv** → falsos negativos (segunda tabla).
   * **near\_threshold.csv** → casos en los que la decisión dependía de muy poca diferencia de score.



## 5. Qué información te aporta este análisis

* Identificar **falsos positivos razonables** → puede que el GOLD esté incompleto, o que el modelo tenga sesgo hacia formulaciones con ciertas keywords.
* Identificar **falsos negativos sistemáticos** → muestra vacíos en el embedding (ej. incapacidad de captar relaciones de sinonimia más profundas).
* Señales como `delta_top1_top2` o `mnn` sirven para decidir si conviene aplicar un **reranker** o introducir un **umbral dinámico**.

# Neighbors

## 1. Estructura de columnas

* **model** → el modelo de embeddings usado (`sentence-transformers/paraphrase-albert-small-v2`).
* **q\_index** → índice numérico de la query (posición en el GOLD o corpus).
* **q\_label** → la query en texto, ej. *“Train personnel on proper operational procedures.”*.
* **rank** → la posición en el ranking para este candidato (1 = más cercano).
* **cand\_label** → el candidato vecino (ej. *“Supervise engineering or other technical personnel.”*).
* **cand\_idx** → índice numérico del candidato en el corpus de DWAs.



### Features numéricas para explicar similitud:

* **sim** → similitud coseno pura entre embeddings de query y candidato (tras normalización L2).

* **z** → z-score de esa similitud dentro de la distribución de scores para esa query:

  $$
  z = \frac{sim - \mu_{q}}{\sigma_{q}}
  $$

  donde $\mu_q$ y $\sigma_q$ son media y desviación estándar de los scores para esa query.
  → Nos dice si el candidato está “significativamente más cerca” que la media.

* **mnn** (mutual nearest neighbor) → `True` si la relación es recíproca (A tiene a B en su top-k y B a A en su top-k).
  → Marca matches más robustos y confiables.

* **lenpen** (length penalty) → penalización proporcional a la diferencia de longitud entre query y candidato.
  → Evita que frases demasiado cortas o largas puntúen artificialmente alto.

* **stopjac** (stopword Jaccard) → Jaccard similarity usando solo stopwords compartidas.
  → Señala si la coincidencia se debe solo a conectores irrelevantes (*and, or, of, with*).

* **bm25** → valor léxico BM25 entre query y candidato (en este caso, todos 0.0 → probablemente no aplicaste el híbrido semántico-léxico aún).

* **score** → score final tras integrar coseno, penalizaciones y filtros:

  $$
  score = sim - \lambda_{len}\cdot lenpen - \rho_{stop}\cdot stopjac + \beta \cdot bm25
  $$

  (dependiendo de tus hiperparámetros).
  Es la métrica usada para ordenar el ranking final.



## 2. Interpretación de las filas

Ejemplo:

* Query: *“Train personnel on proper operational procedures.”*

* Top-1: *“Supervise engineering or other technical personnel.”*

  * sim = 0.52 → relativamente bajo.
  * z = 1.78 → aún así destaca sobre el resto para esta query.
  * mnn = False → el match no es recíproco.
  * score = 0.516 → apenas por encima de 0.5, muestra incertidumbre.

* Top-2: *“Recruit personnel.”*

  * sim = 0.55 (mayor que top-1, interesante), pero penalizado con **lenpen=0.6** (diferencia de longitud).
  * Como el penalizador baja el score a 0.509, cae detrás.
  * Aquí ves cómo los filtros afectan el ranking.

→ Esto ilustra la tensión entre **similitud coseno bruta** y **ajustes heurísticos** (longitud, stopwords, recíproco).

In [5]:
df_neighbors = pd.read_csv("neighbors_all_long.csv")
display(df_neighbors.query("q_label == 'Train personnel on proper operational procedures.'").head(10))


,model,q_index,q_label,rank,cand_label,cand_idx,sim,z,mnn,lenpen,stopjac,bm25,score
0,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,1,Supervise engineering or other technical perso...,12,0.527511,1.783514,False,0.000000,0.111111,0.0,0.516400
1,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,2,Recruit personnel.,114,0.556167,2.364238,True,0.600000,0.166667,0.0,0.509500
2,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,3,Confer with technical personnel to prepare des...,3,0.537444,1.984811,False,0.285714,0.200000,0.0,0.503158
3,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,4,Set up laboratory or field equipment.,211,0.498538,1.196359,False,0.000000,0.000000,0.0,0.498538
4,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,5,Operate industrial equipment.,137,0.518448,1.599856,False,0.400000,0.000000,0.0,0.498448
5,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,6,Hire personnel.,41,0.534525,1.925648,False,0.600000,0.166667,0.0,0.487858
6,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,7,Prepare operational progress or status reports.,44,0.498674,1.199117,False,0.000000,0.111111,0.0,0.487563
7,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,8,Perform human resources activities.,107,0.495107,1.126844,False,0.200000,0.000000,0.0,0.485107
8,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,9,Provide technical guidance to other personnel.,103,0.494916,1.122966,False,0.000000,0.111111,0.0,0.483805
9,sentence-transformers/paraphrase-albert-small-v2,0,Train personnel on proper operational procedures.,10,Confer with other personnel to resolve design ...,62,0.511938,1.467925,False,0.285714,0.200000,0.0,0.477652


## 3. Cómo se llegó a esta tabla

1. En el **bloque D** se definió la función `nearest_neighbors_dwa`:

   * Calcula cosenos entre embeddings query–candidato.
   * Normaliza por query (z-score).
   * Aplica filtros: exclusión de self-match, MNN opcional, length penalty, stopword Jaccard.
   * Combina todo en un **score final**.

2. En el **bloque E**, se ejecutó esa función para cada modelo de la lista `CANDIDATE_MODELS`.

3. Se consolidó en CSV por modelo (`neighbors_<modelo>.csv`) y en un global (`neighbors_all_long.csv`).

4. Lo que ves ahora es **la lista ordenada de vecinos para una query concreta**, con scores y features auxiliares.



## 4. Qué utilidad tiene esta vista

* Permite auditar **por qué un candidato está arriba**: ¿es por coseno alto, por z-score, o a pesar de penalizaciones?
* Muestra **confusión entre candidatos similares** (ej. “Recruit personnel” vs “Train personnel”).
* Ayuda a diseñar **nuevos filtros**: si notas que FP recurrentes tienen `stopjac` alto o `mnn=False`, puedes explotar eso.
* Sirve para entrenar un **reranker**: usar features como `sim`, `z`, `delta_top1_top2`, `mnn`, `lenpen`, `stopjac`, `bm25` como input de un modelo supervisado.

# SIMCSE y TSDAE

##  1. SimCSE (Simple Contrastive Sentence Embedding)

### Idea

* Está basado en **aprendizaje contrastivo**: aprender embeddings que acerquen frases “iguales” y alejen frases diferentes.
* Usa la arquitectura de un Transformer (ej. BERT) pero lo entrena de manera especial.

### Dos variantes

1. **Unsupervised SimCSE**

   * Se usa la misma oración dos veces como “positiva” (simplemente aplicando dropout diferente en el Transformer → produce dos embeddings distintos).
   * Se generan automáticamente pares positivos:

     $$
     (x, x')
     $$

     y pares negativos con las otras frases del batch.
   * Objetivo: maximizar similitud coseno entre $x$ y $x'$, minimizar con el resto.
   * Función de pérdida → **InfoNCE / softmax contrastiva**:

     $$
     L = -\log \frac{\exp(\text{sim}(h_i,h_i^+)/\tau)}{\sum_{j=1}^N \exp(\text{sim}(h_i,h_j)/\tau)}
     $$

     donde $h_i$ = embedding de la frase, $h_i^+$ su positivo (dropout), $\tau$ = temperatura.

2. **Supervised SimCSE**

   * Cuando tienes datasets tipo NLI (entailment/contradiction).
   * Positivos = frases con relación de entailment, negativos = contradiction.

### Ventaja

* Crea embeddings **más robustos semánticamente** porque aprende a diferenciar frases similares de verdad, no solo por tokens.

### Aplicación en tu pipeline

* Entrenaste SimCSE unsupervised sobre tus frases (DWAs + tasks).
* Esperas que los embeddings resulten **más sensibles a matices semánticos** que los SentenceTransformers preentrenados “tal cual”.


##  2. TSDAE (Transformer-based Sequential Denoising AutoEncoder)

### Idea

* Es un **autoencoder** para texto: corrompes la oración de entrada y obligas al modelo a reconstruirla.
* Objetivo: aprender representaciones intermedias (el embedding del encoder) que capturen bien la semántica incluso con ruido.

### Funcionamiento

1. Tomas una oración (ej. una DWA).
2. Le aplicas **ruido**:

   * Borras tokens, cambias orden, etc.
   * Ejemplo: *“Train personnel on proper operational procedures.”* → *“Train … on … procedures.”*
3. Pasas la frase ruidosa por el **encoder (Transformer)** → produce embedding.
4. Un **decoder atado al encoder** debe reconstruir la frase original completa.

   * Loss: cross-entropy entre salida del decoder y la frase original.

$$
L = - \sum_{t=1}^T \log P(y_t \mid \tilde{x}, \theta)
$$

donde $\tilde{x}$ es la frase con ruido, $y_t$ los tokens originales.

### Ventaja

* Al tener que “llenar huecos” y “reconstruir” frases, el encoder aprende embeddings **más generales y robustos a variaciones léxicas**.
* Es útil cuando no tienes pares positivos/negativos (como en SimCSE), pero sí un corpus monolingüe.

### Aplicación en tu pipeline

* Entrenaste TSDAE sobre tus DWAs.
* La idea es que aprenda embeddings que capturen bien la semántica general de tus frases, incluso si están formuladas con ruido o variación léxica.
* Luego usas ese encoder en el paso de similitud semántica (bloques D/E) como un candidato más.


##  3. Diferencias clave

| Aspecto               | SimCSE                                             | TSDAE                                      |
| --------------------- | -------------------------------------------------- | ------------------------------------------ |
| Tipo de entrenamiento | Contrastivo (pares positivos/negativos)            | Autoencoder (reconstrucción)               |
| Datos necesarios      | Corpus de frases (sin etiquetas en versión unsup)  | Corpus de frases                           |
| “Positivos”           | Misma frase con dropout (unsup) o entailment (sup) | La frase original sin ruido                |
| Objetivo              | Aprender a distinguir frases similares/diferentes  | Aprender representaciones robustas a ruido |
| Ventaja               | Captura mejor relaciones finas entre frases        | Embeddings más robustos y generales        |
| Uso en tu pipeline    | Mejor ranking semántico para DWAs cercanas         | Mejor robustez en dominio con pocas frases |



## 4. Cómo encajan en tu workflow

1. Entrenas **SimCSE** o **TSDAE** con tus DWAs y/o tareas.
2. El encoder resultante lo guardas en `results/models/simcse_unsup/` o `tsdae_unsup/`.
3. En el **bloque E** los añades a `CANDIDATE_MODELS`.
4. En el **bloque F** los comparas contra los SentenceTransformers base → ver si suben nDCG\@10, Recall\@1, etc.

